# Ingest: Retail Orders

Digital marketing demo data. Downloads the UCI "Online Retail II" dataset
(1,067,371 line items from a UK-based online gift-ware retailer,
Dec 2009-Dec 2011, public domain — no auth) and persists it as a managed
Spark table.

Confirmed live (2026-08-07) real column names (note: differ slightly
from the dataset's own documentation): `Invoice`, `StockCode`,
`Description`, `Quantity`, `InvoiceDate`, `Price`, `Customer ID` (with a
space — renamed below to avoid needing backticks everywhere downstream),
`Country`. Two sheets (`Year 2009-2010`, `Year 2010-2011`) are unioned
into one table. Includes cancelled orders (`Invoice` prefixed `C`) and
nulls in `Customer ID`/`Description` — realistic messiness, left as-is
here and handled in the transform step.

Requires `openpyxl` (for `pandas.read_excel`) in addition to `pandas`.

Re-runnable: uses `CREATE DATABASE IF NOT EXISTS` and
`mode("overwrite")`, so re-running just refreshes the table.

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "retail"
TABLE_NAME = "orders"
SOURCE_URL = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
LOCAL_ZIP = "/tmp/online_retail_ii.zip"
LOCAL_DIR = "/tmp/online_retail_ii"
SHEET_NAMES = ["Year 2009-2010", "Year 2010-2011"]

## Download and extract source file

In [ ]:
import urllib.request
import zipfile

urllib.request.urlretrieve(SOURCE_URL, LOCAL_ZIP)

with zipfile.ZipFile(LOCAL_ZIP) as zf:
    zf.extractall(LOCAL_DIR)

## Load dataset

Both sheets share the same layout — read each with `pandas`, concatenate,
rename to snake_case, then hand off to Spark.

In [ ]:
xlsx_path = f"{LOCAL_DIR}/online_retail_II.xlsx"

sheet_frames = [pd.read_excel(xlsx_path, sheet_name=sheet) for sheet in SHEET_NAMES]
orders_pdf = pd.concat(sheet_frames, ignore_index=True)

orders_pdf = orders_pdf.rename(columns={
    "Invoice": "invoice_no",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "unit_price",
    "Customer ID": "customer_id",
    "Country": "country",
})

df = spark.createDataFrame(orders_pdf)

## Create schema/database

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}
COMMENT 'Retail order data used for digital marketing demonstrations'
""")

## Persist as a managed Spark table

In [ ]:
df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TABLE_NAME}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_line_items FROM {DATABASE_NAME}.{TABLE_NAME}").show()
spark.sql(f"DESCRIBE {DATABASE_NAME}.{TABLE_NAME}").show(truncate=False)